# 04 — Univariate Feature Testing Framework
## USDJPY H1 · Dukascopy 10yr (62,303 bars · 2016–2026)

Tests each feature's ability to predict forward returns **independently** before combining features in a multi-factor model.

| Metric | Definition | Threshold |
|---|---|---|
| **IC** | Spearman rank corr(feature, fwd_return) | `\|IC\| > 0.05` |
| **IC Decay** | IC vs prediction horizon H | Peak horizon identifies feature lifespan |
| **Rolling IC** | IC in 1000-bar rolling window | IC-IR > 0.5 for stability |
| **Hit Rate** | % bars where sign(feature) == sign(fwd_return) | > 0.5, p < 0.05 |
| **Monotonicity** | Spearman(quintile rank, mean fwd return) | `\|ρ\| > 0.8` |

**Reject criteria:** ① Non-stationary (ADF p ≥ 0.05) OR ② Peak |IC| < 0.05 across all horizons.

## Section 0 — Imports & Configuration

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# Add project root to path
PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJ_ROOT not in sys.path:
    sys.path.insert(0, PROJ_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import matplotlib.ticker as mticker

from src.data.loader import FXDataLoader
from src.features.generators import (
    atr,
    hp_trend_slope,
    hp_trend_curvature,
    ma_spread_on_trend,
    trend_deviation_from_ma,
    vol_regime,
)
from src.features.library import FeatureLibrary
from src.features.returns import compute_log_returns
from src.features.testing import (
    UnivariateFeatureTester,
    compute_forward_returns,
    check_stationarity,
    compute_ic,
    compute_ic_decay,
    compute_rolling_ic,
    compute_hit_rate,
    compute_quantile_ic,
    IC_THRESHOLD,
)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SYMBOL         = "USDJPY_10yr_1h_dukascopy"
HP_LAMBDA      = 3.9e9   # Theoretically calibrated for H1 FX (≈ 1600 × (504/12)^4)
HP_WINDOW      = 500     # Bars per rolling causal HP window
T1             = 72      # Short MA for ma_spread_on_trend
T2             = 240     # Long MA for ma_spread_on_trend
TREND_DEV_WIN  = 240     # MA window for trend_deviation_from_ma
ATR_WINDOW     = 20      # ATR normaliser window
HORIZONS       = [1, 4, 12, 24, 48]   # Forward lookup horizons (H1 bars)
ROLLING_WIN    = 1000    # Rolling IC window  ≈ 6 months H1
N_QUANTILES    = 5       # Quintile bins for monotonicity analysis

HP_CACHE_PATH  = os.path.join(PROJ_ROOT, "data", "interim", "hp_trends_window500.csv")
RAW_DATA_PATH  = os.path.join(PROJ_ROOT, "data", "raw")

print(f"Symbol         : {SYMBOL}")
print(f"HP Lambda      : {HP_LAMBDA:.2e}")
print(f"MA periods     : T1={T1}, T2={T2}  |  trend_dev window={TREND_DEV_WIN}")
print(f"Horizons (bars): {HORIZONS}")
print(f"Rolling window : {ROLLING_WIN} bars  ≈ {ROLLING_WIN/24:.0f} trading days H1")
print(f"IC threshold   : {IC_THRESHOLD}")

## Section 1 — Data Loading

In [ ]:
loader = FXDataLoader(RAW_DATA_PATH)
df     = loader.load(SYMBOL)

close  = df["close"]
high   = df["high"]
low    = df["low"]

print(f"Loaded : {len(df):,} bars")
print(f"Range  : {df.index[0]} → {df.index[-1]}")
print(f"Columns: {list(df.columns)}")
df.tail(3)

In [ ]:
# Load pre-computed HP-trend cache (avoids ~6 min re-computation per lambda)
hp_cache = pd.read_csv(HP_CACHE_PATH, index_col=0, parse_dates=True)

# Align timezone
if hp_cache.index.tz is None:
    hp_cache.index = hp_cache.index.tz_localize("UTC")
hp_cache = hp_cache.reindex(df.index)

print(f"HP cache : {len(hp_cache):,} rows × {hp_cache.shape[1]} cols")
print(f"Columns  : {list(hp_cache.columns)}")

# λ=3.9B — theoretically calibrated for H1 FX (used for slope / curvature)
LAMBDA_COL_DERIV = "hp_trend_3.9B"
hp_trend_hi = hp_cache[LAMBDA_COL_DERIV]

# λ=1B — empirically used in strategy_b_efc for mean-reversion quality gate.
# At λ=3.9B the trend is extremely smooth, so trend_dev_from_ma has near-zero
# variance and near-zero IC. λ=1B is less aggressively smoothed and produces
# meaningful overextension readings.
LAMBDA_COL_MR = "hp_trend_1B"
hp_trend_mr = hp_cache[LAMBDA_COL_MR]

print(f"\n{LAMBDA_COL_DERIV!r} (derivatives): {hp_trend_hi.notna().sum():,} valid")
print(f"{LAMBDA_COL_MR!r}  (mean-reversion): {hp_trend_mr.notna().sum():,} valid")

## Section 2 — Feature Computation

Nine features are computed:

| # | Feature | Type | Source |
|---|---------|------|--------|
| 1 | `hp_trend_slope` | HP momentum | `generators.hp_trend_slope` |
| 2 | `hp_trend_curvature` | HP exhaustion | `generators.hp_trend_curvature` |
| 3 | `trend_dev_from_ma` | Mean reversion on S* | `generators.trend_deviation_from_ma` |
| 4 | `ma_spread_on_trend` | Trend strength | `generators.ma_spread_on_trend` |
| 5 | `momentum_24h` | Raw log-return momentum | `FeatureLibrary.momentum` |
| 6 | `rsi_14` | RSI oscillator | `FeatureLibrary.rsi` |
| 7 | `distance_from_ma` | Z-scored MA deviation | `FeatureLibrary.distance_from_ma` |
| 8 | `lambda_sensitivity` | Cross-λ disagreement | Computed from cache |
| 9 | `vol_regime` | Volatility regime (0/1/2) | `generators.vol_regime` ⚠ categorical |

> **Note:** `vol_regime` is categorical — excluded from IC tests; analysed separately in Section 8.

In [ ]:
log_ret  = compute_log_returns(close)
atr_s    = atr(high, low, close, window=ATR_WINDOW)

# ── HP derivatives: λ=3.9B (theoretically calibrated for H1 FX) ─────────────
slope    = hp_trend_slope(hp_trend_hi)
slope.name = "hp_trend_slope"

curv     = hp_trend_curvature(hp_trend_hi)
curv.name = "hp_trend_curvature"

# ── Mean-reversion HP features: λ=1B (empirically preferred for overextension) ─
# Note: at λ=3.9B the HP trend is so smooth that trend_dev_from_ma ≈ 0 (no signal);
# λ=1B allows the trend to deviate from its own MA in a meaningful way.
trend_dev = trend_deviation_from_ma(hp_trend_mr, window=TREND_DEV_WIN, atr_series=atr_s)
trend_dev.name = "trend_dev_from_ma"

ma_spread = ma_spread_on_trend(hp_trend_mr, t1=T1, t2=T2, atr_series=atr_s)
ma_spread.name = "ma_spread_on_trend"

# ── FeatureLibrary features (on raw close prices) ────────────────────────────
mom      = FeatureLibrary.momentum(close, period=24)      # ~1 trading day H1
mom.name = "momentum_24h"

rsi_feat = FeatureLibrary.rsi(close, period=14)
rsi_feat.name = "rsi_14"

dist_ma  = FeatureLibrary.distance_from_ma(close, window=20)
dist_ma.name = "distance_from_ma"

# ── Lambda sensitivity (cross-λ slope std from cache) ────────────────────────
# Slope = first diff of each lambda's HP trend; std across lambdas = disagreement
hp_slopes    = hp_cache.diff()
lambda_sens  = (hp_slopes.std(axis=1) / atr_s).replace([np.inf, -np.inf], np.nan)
lambda_sens.name = "lambda_sensitivity"

# ── Volatility regime (categorical — kept separate) ──────────────────────────
vol_reg  = vol_regime(log_ret)
vol_reg.name = "vol_regime"

# ── All continuous features in one dict (for UnivariateFeatureTester) ────────
FEATURES = {
    "hp_trend_slope":    slope,      # λ=3.9B
    "hp_trend_curvature": curv,      # λ=3.9B
    "trend_dev_from_ma": trend_dev,  # λ=1B
    "ma_spread_on_trend": ma_spread, # λ=1B
    "momentum_24h":      mom,
    "rsi_14":            rsi_feat,
    "distance_from_ma":  dist_ma,
    "lambda_sensitivity": lambda_sens,
}

# Summary
print(f"{'Feature':<22}  {'Lambda':<8}  {'Valid':>8}  {'NaN':>6}")
print("-" * 52)
lambdas = {
    "hp_trend_slope": "3.9B", "hp_trend_curvature": "3.9B",
    "trend_dev_from_ma": "1B", "ma_spread_on_trend": "1B",
}
for name, feat in FEATURES.items():
    lam = lambdas.get(name, "raw")
    print(f"{name:<22}  {lam:<8}  {feat.notna().sum():>8,}  {feat.isna().sum():>6,}")
print(f"\nvol_regime (categorical): {vol_reg.notna().sum():,} valid")

## Section 3 — Stationarity Tests

ADF test for unit roots. **Reject criterion #1**: non-stationary features (ADF p ≥ 0.05) are excluded from backtesting regardless of apparent IC.

> **FX context**: HP trend *level* S* is I(1) (non-stationary) on large datasets — already confirmed in Step 1 of the research pipeline. ΔS* (slope) and Δ²S* (curvature) are first/second differences and expected to be I(0).

In [ ]:
adf_results = {}
for name, feat in FEATURES.items():
    adf_results[name] = check_stationarity(feat, name)

# Display as table
rows = []
for name, r in adf_results.items():
    rows.append({
        "Feature":        r.feature_name,
        "ADF Statistic":  f"{r.adf_stat:.4f}",
        "p-value":        f"{r.p_value:.6f}",
        "Lags (AIC)":     r.n_lags,
        "N obs":          f"{r.n_obs:,}",
        "Stationary?":    "✓ YES" if r.is_stationary else "✗ NO",
    })

adf_df = pd.DataFrame(rows).set_index("Feature")
print("ADF Stationarity Tests  (H0: unit root, reject at p < 0.05)")
print("=" * 72)
adf_df

## Section 4 — IC Decay Curves

IC (Spearman rank correlation) between each feature and forward log-returns at horizons H ∈ {1, 4, 12, 24, 48}.

- **Steep decay** → short-lived signal (intraday / next-bar)
- **Flat decay** → persistent signal (multi-day carry)
- **Sign flip** → signal reverses at longer horizons (common in FX mean-reversion)

The **peak-IC horizon** determines the feature's natural hold period.

In [ ]:
ic_decay_results = {}
for name, feat in FEATURES.items():
    ic_decay_results[name] = compute_ic_decay(feat, close, HORIZONS, name)
    d = ic_decay_results[name]
    print(f"{name:<22}  peak IC={d.peak_ic:+.4f} at H={d.peak_horizon:>2}  "
          f"|IC|={d.peak_abs_ic:.4f}  "
          f"{'PASS' if d.peak_abs_ic >= IC_THRESHOLD else 'FAIL ← below threshold'}")

In [ ]:
n_feat = len(FEATURES)
ncols  = 2
nrows  = (n_feat + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(13, nrows * 3.2), constrained_layout=True)
axes = axes.flatten()

for idx, (name, decay) in enumerate(ic_decay_results.items()):
    ax  = axes[idx]
    ics = np.array(decay.ic_values, dtype=float)
    pvs = np.array(decay.p_values, dtype=float)

    colors = ["#2ca02c" if v > 0 else "#d62728" for v in ics]
    ax.bar(decay.horizons, ics, color=colors, alpha=0.8, width=2.5)

    # Significance markers
    for h, ic_val, pv in zip(decay.horizons, ics, pvs):
        if not np.isnan(pv) and pv < 0.05:
            marker = "***" if pv < 0.001 else ("**" if pv < 0.01 else "*")
            y_off  = 0.003 if ic_val >= 0 else -0.003
            va     = "bottom" if ic_val >= 0 else "top"
            ax.annotate(marker, (h, ic_val + y_off), ha="center", va=va,
                        fontsize=7, color="black")

    # Reference lines
    ax.axhline(0,            color="black", linewidth=0.7)
    ax.axhline( IC_THRESHOLD, color="steelblue", linewidth=0.9, linestyle="--",
                label=f"+{IC_THRESHOLD}")
    ax.axhline(-IC_THRESHOLD, color="steelblue", linewidth=0.9, linestyle="--",
                label=f"-{IC_THRESHOLD}")

    ax.set_title(f"{name}\npeak IC={decay.peak_ic:+.4f} @ H={decay.peak_horizon}", pad=4)
    ax.set_xlabel("Horizon (bars)"); ax.set_ylabel("IC (Spearman ρ)")
    ax.set_xticks(decay.horizons)
    ax.set_xlim(-2, max(decay.horizons) + 3)

# Hide extra subplots
for idx in range(n_feat, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("IC Decay Curves — USDJPY H1 (Full Sample)", fontsize=14, fontweight="bold")
plt.show()

## Section 5 — Rolling IC Stability

1,000-bar rolling Spearman IC (≈ 6 months H1) at each feature's **peak-IC horizon**.

- Green band = rolling IC above +IC_THRESHOLD
- Red band = rolling IC below -IC_THRESHOLD  
- **IC-IR** (IC Information Ratio = mean/std) > 0.5 → reliable signal; < 0.3 → regime-dependent

⚠ *This section takes ~1–2 seconds per feature (exact Spearman per rolling window).*

In [ ]:
rolling_ic_results = {}
for name, feat in FEATURES.items():
    peak_h   = ic_decay_results[name].peak_horizon
    fwd      = compute_forward_returns(close, peak_h)
    rolling_ic_results[name] = compute_rolling_ic(feat, fwd, ROLLING_WIN, name)
    r = rolling_ic_results[name]
    print(f"{name:<22}  mean IC={r.mean_ic:+.4f}  std={r.std_ic:.4f}  "
          f"IC-IR={r.ic_ir:+.3f}  pct_pos={r.pct_positive:.1%}  "
          f"pct>thr={r.pct_above_threshold:.1%}")

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(13, nrows * 3.0), constrained_layout=True)
axes = axes.flatten()

for idx, (name, ric) in enumerate(rolling_ic_results.items()):
    ax      = axes[idx]
    ic_s    = ric.series.dropna()
    peak_h  = ic_decay_results[name].peak_horizon

    if len(ic_s) == 0:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center",
                transform=ax.transAxes)
        continue

    ax.plot(ic_s.index, ic_s.values, color="steelblue", linewidth=0.8, alpha=0.9)
    ax.fill_between(ic_s.index, ic_s.values, 0,
                    where=ic_s.values > IC_THRESHOLD,
                    color="#2ca02c", alpha=0.25, label=f"IC > {IC_THRESHOLD}")
    ax.fill_between(ic_s.index, ic_s.values, 0,
                    where=ic_s.values < -IC_THRESHOLD,
                    color="#d62728", alpha=0.25, label=f"IC < -{IC_THRESHOLD}")

    ax.axhline(0,             color="black",     linewidth=0.7)
    ax.axhline( IC_THRESHOLD, color="#2ca02c",   linewidth=0.8, linestyle="--")
    ax.axhline(-IC_THRESHOLD, color="#d62728",   linewidth=0.8, linestyle="--")

    ax.set_title(
        f"{name}  (H={peak_h})\n"
        f"mean={ric.mean_ic:+.3f}  IC-IR={ric.ic_ir:+.3f}  "
        f"pct>{IC_THRESHOLD:.2f}={ric.pct_above_threshold:.0%}",
        pad=3,
    )
    ax.set_ylabel("Rolling IC"); ax.yaxis.set_major_locator(mticker.MaxNLocator(5))
    # Sparse x-tick
    years = pd.date_range(ic_s.index.min(), ic_s.index.max(), freq="YS")
    ax.set_xticks(years); ax.set_xticklabels([y.year for y in years], rotation=45)

for idx in range(n_feat, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    f"1,000-bar Rolling IC Stability — USDJPY H1  (window={ROLLING_WIN} bars)",
    fontsize=14, fontweight="bold",
)
plt.show()

## Section 6 — Hit Rate Analysis

**Hit rate** = fraction of bars where `sign(feature) == sign(fwd_return)`.  
Tested across all horizons; assessed against `H0: hit_rate = 0.5` using binomial test.

A feature can have meaningful IC but a hit rate at chance level if the IC is driven by large moves (rank-based correlation rewards magnitude, not just direction).

In [ ]:
hit_rate_matrix = {}   # name → {horizon: HitRateResult}
for name, feat in FEATURES.items():
    hit_rate_matrix[name] = {}
    for h in HORIZONS:
        fwd = compute_forward_returns(close, h)
        hit_rate_matrix[name][h] = compute_hit_rate(feat, fwd, name, h)

# Build display arrays
hr_vals = np.full((n_feat, len(HORIZONS)), np.nan)
hr_sig  = np.zeros((n_feat, len(HORIZONS)), dtype=bool)
for i, name in enumerate(FEATURES):
    for j, h in enumerate(HORIZONS):
        r = hit_rate_matrix[name][h]
        hr_vals[i, j] = r.hit_rate
        hr_sig[i, j]  = r.is_significant

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 0.7 * n_feat + 1.5), constrained_layout=True)
norm    = TwoSlopeNorm(vcenter=0.5, vmin=0.48, vmax=0.52)
im      = ax.imshow(hr_vals, cmap="RdYlGn", norm=norm, aspect="auto")

plt.colorbar(im, ax=ax, label="Hit Rate (0.5 = random)")
ax.set_xticks(range(len(HORIZONS))); ax.set_xticklabels([f"H={h}" for h in HORIZONS])
ax.set_yticks(range(n_feat)); ax.set_yticklabels(list(FEATURES.keys()))
ax.set_xlabel("Horizon"); ax.set_title("Hit Rate by Feature × Horizon\n(* = significant vs 0.5, binomial test p<0.05)", fontweight="bold")

for i in range(n_feat):
    for j in range(len(HORIZONS)):
        val = hr_vals[i, j]
        if not np.isnan(val):
            label = f"{val:.3f}" + ("\n*" if hr_sig[i, j] else "")
            ax.text(j, i, label, ha="center", va="center", fontsize=8)
plt.show()

## Section 7 — Quantile Monotonicity

Mean forward return (bps) per **quintile** of feature values.

A good feature should show a **monotonic** relationship: higher feature value → higher (or consistently lower) forward return across all quantiles. Spearman(bin_rank, mean_return) tests this ordering: |ρ| > 0.8 with p < 0.05 confirms monotonicity.

In [ ]:
mono_results = {}
for name, feat in FEATURES.items():
    peak_h = ic_decay_results[name].peak_horizon
    fwd    = compute_forward_returns(close, peak_h)
    mono_results[name] = compute_quantile_ic(feat, fwd, N_QUANTILES, name)
    m = mono_results[name]
    print(f"{name:<22}  ρ={m.spearman_rho:+.3f}  p={m.p_value:.4f}  "
          f"{'MONOTONIC ✓' if m.is_monotonic else 'not monotonic'}")

# ── Plot quintile bar charts ──────────────────────────────────────────────────
fig, axes = plt.subplots(nrows, ncols, figsize=(13, nrows * 3.0), constrained_layout=True)
axes = axes.flatten()

for idx, (name, mono) in enumerate(mono_results.items()):
    ax      = axes[idx]
    labels  = mono.quantile_labels
    returns = mono.mean_fwd_returns_bps
    peak_h  = ic_decay_results[name].peak_horizon

    if not labels:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(name); continue

    colors = ["#d62728" if v < 0 else "#2ca02c" for v in returns]
    ax.bar(labels, returns, color=colors, alpha=0.85, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(
        f"{name}  (H={peak_h})\n"
        f"monotonicity ρ={mono.spearman_rho:+.3f}  p={mono.p_value:.3f}  "
        f"{'✓' if mono.is_monotonic else '✗'}",
        pad=3,
    )
    ax.set_ylabel("Mean fwd return (bps)")
    ax.set_xlabel("Quintile (Q1=low, Q5=high)")

for idx in range(n_feat, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    f"Quintile Monotonicity — USDJPY H1  (at each feature's peak-IC horizon)",
    fontsize=14, fontweight="bold",
)
plt.show()

## Section 8 — Volatility Regime Special Analysis

`vol_regime` is categorical (0=Low, 1=Med, 2=High) — IC and hit rate are not meaningful.  
Instead: compare mean forward returns and mean IC of other features *per regime*.

In [ ]:
REGIME_LABELS = {0: "Low Vol", 1: "Med Vol", 2: "High Vol"}
REGIME_COLORS = {0: "#4878cf", 1: "#6acc65", 2: "#d65f5f"}

# ── 1. Mean forward return per regime at each horizon ────────────────────────
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(14, 3.5), constrained_layout=True)

for ax, h in zip(axes, HORIZONS):
    fwd      = compute_forward_returns(close, h)
    combined = pd.concat([vol_reg, fwd], axis=1).dropna()
    combined.columns = ["regime", "fwd"]
    regime_means = combined.groupby("regime")["fwd"].agg(["mean", "sem"]) * 10_000

    regimes = sorted(regime_means.index)
    ys      = [regime_means.loc[r, "mean"] for r in regimes]
    errs    = [regime_means.loc[r, "sem"]  for r in regimes]
    labels  = [REGIME_LABELS.get(r, str(r)) for r in regimes]
    colors  = [REGIME_COLORS.get(r, "grey") for r in regimes]
    ns      = [combined[combined["regime"] == r].shape[0] for r in regimes]

    ax.bar(labels, ys, color=colors, alpha=0.8, yerr=errs, capsize=4, error_kw={"linewidth": 1.2})
    ax.axhline(0, color="black", linewidth=0.7)
    ax.set_title(f"H={h}")
    ax.set_ylabel("Mean fwd return (bps)" if h == HORIZONS[0] else "")
    for i, (lbl, n) in enumerate(zip(labels, ns)):
        ax.text(i, min(ys) - 0.3, f"n={n:,}", ha="center", fontsize=7)

fig.suptitle("Mean Forward Return by Volatility Regime — USDJPY H1", fontsize=13, fontweight="bold")
plt.show()

# ── 2. IC per regime for best feature (trend_dev_from_ma) ───────────────────
print("\n─── IC per vol_regime  (feature: trend_dev_from_ma) ───")
for h in HORIZONS:
    fwd = compute_forward_returns(close, h)
    combined = pd.concat([vol_reg, trend_dev, fwd], axis=1).dropna()
    combined.columns = ["regime", "feature", "fwd"]
    print(f"\n  H={h}")
    for reg_val in sorted(combined["regime"].unique()):
        sub = combined[combined["regime"] == reg_val]
        if len(sub) < 30:
            continue
        from scipy.stats import spearmanr
        rho, pv = spearmanr(sub["feature"], sub["fwd"])
        stars = "***" if pv < 0.001 else ("**" if pv < 0.01 else ("*" if pv < 0.05 else " "))
        print(f"    {REGIME_LABELS[int(reg_val)]:<10}: IC={rho:+.4f}  p={pv:.4f} {stars}  n={len(sub):,}")

## Section 9 — Batch Summary & Pass/Fail Verdict

`UnivariateFeatureTester.test_all()` runs the complete pipeline and applies both reject criteria:
1. **Non-stationary** (ADF p ≥ 0.05)
2. **Peak |IC| < 0.05** across all horizons

In [ ]:
tester  = UnivariateFeatureTester(
    prices         = close,
    ic_threshold   = IC_THRESHOLD,
    horizons       = HORIZONS,
    rolling_window = ROLLING_WIN,
)
summary_df = tester.test_all(FEATURES)

# Pretty display
display_cols = [
    "name", "is_stationary", "adf_p",
    "peak_ic", "peak_abs_ic", "peak_horizon",
    "rolling_ic_mean", "ic_ir",
    "hit_rate", "monotonicity_rho",
    "passed", "reject_reason",
]
disp = summary_df[display_cols].copy()

def fmt_float(v, fmt=".4f"):
    return f"{v:{fmt}}" if isinstance(v, float) and not np.isnan(v) else str(v)

for col in ["adf_p", "peak_ic", "peak_abs_ic", "rolling_ic_mean", "ic_ir",
            "hit_rate", "monotonicity_rho"]:
    disp[col] = disp[col].apply(lambda v: fmt_float(v))

disp["passed"] = disp["passed"].map({True: "✓ PASS", False: "✗ FAIL"})

print("\n" + "=" * 100)
print("UNIVARIATE FEATURE TEST SUMMARY — USDJPY H1 (Dukascopy 10yr)".center(100))
print("=" * 100)
print(disp.to_string(index=False))
print("=" * 100)

In [ ]:
passed = summary_df[summary_df["passed"]]
failed = summary_df[~summary_df["passed"]]

print(f"{'='*60}")
print(f"  PASSED ({len(passed)}/{len(summary_df)}) — Proceed to backtesting")
print(f"{'='*60}")
for _, row in passed.iterrows():
    print(f"  ✓  {row['name']:<22}  peak IC={row['peak_ic']:+.4f} @ H={row['peak_horizon']}"
          f"  IC-IR={row['ic_ir']:+.3f}  hit={row['hit_rate']:.3f}")

print(f"\n{'='*60}")
print(f"  FAILED ({len(failed)}/{len(summary_df)}) — Rejected")
print(f"{'='*60}")
for _, row in failed.iterrows():
    print(f"  ✗  {row['name']:<22}  → {row['reject_reason']}")

print(f"\n{'─'*60}")
print("  Note: vol_regime is categorical → excluded from IC-based testing")
print("        Use as regime filter in multi-feature backtests.")